# The Dropout Bug That Only Shows Up at Test Time

**Topic:** Dropout &nbsp;·&nbsp; **Difficulty:** Medium

Dropout zeroes a fraction `p` of activations during training, which scales down
the expected value of everything downstream by `1 - p`. If you do not correct for
that, the network you evaluate at test time (with all units active) sees inputs
that are `1/(1-p)` times larger than anything it was trained on.

Implement both conventions and measure the mismatch. Inverted dropout scales
during training so test time needs no special case at all, which is why every
framework uses it.

## The data (created in the next cell)

- `H` `(4000, 256)`: a layer of positive activations.
- `p = 0.4` dropout probability.

## Your task

With **NumPy only**:

1. Draw the mask exactly as
   `keep = (np.random.default_rng(77).random(H.shape) >= p).astype(float)`,
   since the grader compares your four numbers bit for bit: a faithful but
   differently-drawn mask shifts the training means by about 5e-4, which is
   far outside the tolerance. Then `vanilla_train = H * keep`, which zeroes
   each unit with probability `p`, and do **not** rescale.
2. `inverted_train`: zero with probability `p` and divide the survivors by
   `1 - p`.
3. At test time the activations are just `H` for inverted dropout, and
   `H * (1 - p)` for the vanilla convention (the classic correction, not graded).
4. Set `answers` to a dict with:
   - `"mean_train_vanilla"`, `"mean_train_inverted"`: mean activation during
     training under each,
   - `"mean_test_inverted"`: mean of the inverted-dropout test activations,
   - `"drift_if_uncorrected"`: the ratio `mean(H) / mean_train_vanilla`, the
     factor by which test activations would be inflated if you forgot the
     correction entirely.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks inverted dropout matches its own test-time statistics and that the
uncorrected drift is close to `1/(1-p)`.

In [ ]:
import numpy as np

rng = np.random.default_rng(2424)
H = np.abs(rng.normal(1.0, 0.4, (4000, 256)))
p = 0.4
print(f"H {H.shape}, mean activation {H.mean():.4f}, dropout p = {p}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("mean_train_vanilla", [], 0.6013179474212244, -0.10068348338853919, 1e-06),
    ("mean_train_inverted", [], 1.002196579035374, -0.1678058056475653, 1e-06),
    ("mean_test_inverted", [], 1.0021448235247945, -0.16779713980761818, 1e-06),
    ("drift_if_uncorrected", [], 1.6665805965421987, -0.2790489466133895, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "drift_if_uncorrected" in answers:
    _match = abs(answers["mean_train_inverted"] - answers["mean_test_inverted"]) < 0.02
    _drift = abs(answers["drift_if_uncorrected"] - 1.0 / (1 - p)) < 0.05
    print(f"  {'inverted matches test':<24} {'ok' if _match else 'wrong'}")
    print(f"  {'drift is 1/(1-p)':<24} {'ok' if _drift else 'wrong'}  "
          f"({answers['drift_if_uncorrected']:.3f} vs {1/(1-p):.3f})")
    _ok = _ok and _match and _drift

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The expected value of a dropped-and-unscaled activation is `(1-p) * h`, so
training statistics are systematically smaller than test statistics by a factor
of `1 - p` (here 0.6x), i.e. test values are `1/(1-p)`, about 1.67x,
larger. Vanilla dropout fixes it by scaling at *test* time,
which means inference has to know whether dropout was used. Inverted dropout
scales at *train* time, so the deployed graph is unchanged and `model.eval()` is
free. Getting this backwards produces a model that works in training and is
subtly miscalibrated in production.

```python
import numpy as np

rng2 = np.random.default_rng(77)
keep = (rng2.random(H.shape) >= p).astype(float)

vanilla_train = H * keep
inverted_train = H * keep / (1 - p)

test_inverted = H

answers = {
    "mean_train_vanilla": float(vanilla_train.mean()),
    "mean_train_inverted": float(inverted_train.mean()),
    "mean_test_inverted": float(test_inverted.mean()),
    "drift_if_uncorrected": float(H.mean() / vanilla_train.mean()),
}
```